# 업무 근거 쿼리 생성 + 업무 추출 멀티에이전트

이 노트북은 기존 단일 목 문서 입력 방식을 `추출 에이전트 → 쿼리 생성 에이전트 → VectorDB 검색 → 근거 보강 → 추출 에이전트` 루프로 교체하는 실행 가능한 설계 예제다.

- 기본 실행은 API 비용이 없는 Mock 모드다.
- `USE_OPENAI=True`와 `OPENAI_API_KEY`를 설정하면 OpenAI Responses API의 Structured Outputs를 사용한다.
- `InMemoryVectorRetriever`만 실제 pgvector/Retriever Adapter로 교체하면 그래프 구조는 유지된다.


In [ ]:
# 필요 시 한 번만 실행
# %pip install -U openai pydantic langgraph typing_extensions


In [ ]:
from __future__ import annotations

import os
import re
import uuid
from collections import defaultdict
from typing import Annotated, Any, Literal, Protocol, TypedDict

from pydantic import BaseModel, Field
from typing_extensions import NotRequired
from langgraph.graph import END, START, StateGraph

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6")
USE_OPENAI = os.getenv("USE_OPENAI", "false").lower() == "true"
MAX_ATTEMPTS_PER_STAGE = 2
MAX_TOTAL_CHUNKS = 20


## 1. 데이터 계약

최종 업무는 반드시 `evidence_chunk_ids`를 가지며, 근거가 없는 값은 `None`, 빈 배열, `missing_fields`로 표현한다.


In [ ]:
StageName = Literal["TASK_DISCOVERY", "TASK_CORE", "ASSIGNMENT_REQUIREMENT", "EXECUTION_CONDITION", "FINAL_EXTRACTION"]
Intent = Literal["TASK_DISCOVERY", "TASK_CORE", "ASSIGNMENT_REQUIREMENT", "EXECUTION_CONDITION"]
Action = Literal["SEARCH", "ADVANCE", "FINALIZE"]


class RetrievedChunk(BaseModel):
    document_id: str
    chunk_id: str
    sequence: int
    text: str
    contextualized_text: str = ""
    heading_path: list[str] = Field(default_factory=list)
    page_numbers: list[int] = Field(default_factory=list)
    retrieval_score: float = 0.0
    quality_flags: list[str] = Field(default_factory=list)
    intent: Intent = "TASK_CORE"


class SearchNeed(BaseModel):
    stage: StageName
    intent: Intent
    task_hint: str = ""
    missing_information: list[str]
    reason: str


class QueryPlan(BaseModel):
    query_id: str
    intent: Intent
    queries: list[str] = Field(min_length=1, max_length=3)
    top_k: int = Field(default=5, ge=1, le=10)
    context_expansion: list[Literal["same_heading", "previous_chunk", "next_chunk"]] = Field(
        default_factory=lambda: ["same_heading", "previous_chunk", "next_chunk"]
    )


class ExtractedTask(BaseModel):
    task_id: str
    title: str
    description: str
    required_role: str | None = None
    required_skills: list[str] = Field(default_factory=list)
    effort_hours: float | None = None
    start_date: str | None = None
    due_date: str | None = None
    priority: str | None = None
    dependencies: list[str] = Field(default_factory=list)
    constraints: list[str] = Field(default_factory=list)
    risks: list[str] = Field(default_factory=list)
    acceptance_criteria: list[str] = Field(default_factory=list)
    deliverables: list[str] = Field(default_factory=list)
    missing_fields: list[str] = Field(default_factory=list)
    evidence_chunk_ids: list[str] = Field(min_length=1)


class ExtractionDecision(BaseModel):
    action: Action
    rationale: str
    search_need: SearchNeed | None = None
    tasks: list[ExtractedTask] = Field(default_factory=list)


STAGES: list[dict[str, Any]] = [
    {"name": "TASK_DISCOVERY", "intent": "TASK_DISCOVERY", "required": ["기획서에 명시된 수행 업무 후보", "업무를 뒷받침하는 직접 문장"]},
    {"name": "TASK_CORE", "intent": "TASK_CORE", "required": ["명시적 수행 업무", "요구사항", "산출물 또는 완료 기준"]},
    {"name": "ASSIGNMENT_REQUIREMENT", "intent": "ASSIGNMENT_REQUIREMENT", "required": ["담당 역할", "필수 기술 또는 경험"]},
    {"name": "EXECUTION_CONDITION", "intent": "EXECUTION_CONDITION", "required": ["공수", "일정·마감일", "우선순위·의존성·제약·위험"]},
    {"name": "FINAL_EXTRACTION", "intent": None, "required": []},
]


In [ ]:
class AgentState(TypedDict):
    project_id: str
    primary_document_id: str
    primary_document_title: str
    document_ids: list[str]
    analysis_goal: str
    stage_index: int
    stage_attempts: dict[str, int]
    evidence_chunks: list[dict[str, Any]]
    query_history: list[str]
    extracted_tasks: list[dict[str, Any]]
    warnings: list[str]
    trace: list[str]
    max_attempts_per_stage: int
    max_total_chunks: int
    search_need: NotRequired[dict[str, Any] | None]
    query_plan: NotRequired[dict[str, Any] | None]
    latest_chunks: NotRequired[list[dict[str, Any]]]
    action: NotRequired[Action]
    workloads: NotRequired[list[dict[str, Any]]]
    candidate_results: NotRequired[list[dict[str, Any]]]
    final_result: NotRequired[dict[str, Any]]


## 2. 쿼리 생성 에이전트와 업무 추출 에이전트

OpenAI 모드는 Responses API의 `responses.parse(..., text_format=PydanticModel)`로 출력 스키마를 강제한다. Mock 모드는 그래프 구조 검증용이다.


In [ ]:
def evidence_for_stage(state: AgentState, intent: str | None) -> list[dict[str, Any]]:
    if intent is None:
        return state["evidence_chunks"]
    return [c for c in state["evidence_chunks"] if c.get("intent") == intent]


class QueryAgent(Protocol):
    def generate(self, state: AgentState, need: SearchNeed) -> QueryPlan: ...


class ExtractionAgent(Protocol):
    def decide(self, state: AgentState) -> ExtractionDecision: ...


class OpenAIQueryAgent:
    def __init__(self, model: str = MODEL):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model

    def generate(self, state: AgentState, need: SearchNeed) -> QueryPlan:
        response = self.client.responses.parse(
            model=self.model,
            input=[
                {"role": "system", "content": (
                    "당신은 VectorDB 업무 근거 검색어 생성 전담 에이전트다. "
                    "업무를 추출하거나 답을 만들지 말고, 요청받은 부족 정보를 찾을 질의만 만든다. "
                    "질의는 한국어 중심의 핵심 명사·동작·산출물·조건을 포함하고, 이미 사용한 질의를 반복하지 않는다."
                )},
                {"role": "user", "content": (
                    f"검색 필요 정보: {need.model_dump_json(ensure_ascii=False)}\n"
                    f"사용자 선택 대표 기획서: {state['primary_document_id']} / {state['primary_document_title']}\n"
                    f"전체 검색 가능 문서 범위: {state['document_ids']}\n"
                    f"이미 사용한 질의: {state['query_history']}"
                )},
            ],
            text_format=QueryPlan,
        )
        return response.output_parsed


class OpenAIExtractionAgent:
    def __init__(self, model: str = MODEL):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model

    def decide(self, state: AgentState) -> ExtractionDecision:
        stage = STAGES[state["stage_index"]]
        evidence = evidence_for_stage(state, stage["intent"])
        attempts = state["stage_attempts"].get(stage["name"], 0)
        response = self.client.responses.parse(
            model=self.model,
            input=[
                {"role": "system", "content": (
                    "당신은 근거 기반 업무 추출 오케스트레이터다. 현재 단계만 평가한다. "
                    "근거가 없거나 부족하면 SEARCH와 구체적인 SearchNeed를 반환한다. "
                    "현재 단계가 충분하면 ADVANCE를 반환한다. FINAL_EXTRACTION에서는 FINALIZE와 업무 목록을 반환한다. "
                    "제공된 Chunk에 없는 역할, 기술, 공수, 날짜를 추정하지 않는다. "
                    "최종 업무에는 직접 근거인 evidence_chunk_ids가 반드시 있어야 한다."
                )},
                {"role": "user", "content": (
                    f"현재 단계: {stage}\n검색 시도: {attempts}/{state['max_attempts_per_stage']}\n"
                    f"현재 단계 근거: {evidence}\n전체 누적 근거: {state['evidence_chunks']}"
                )},
            ],
            text_format=ExtractionDecision,
        )
        return response.output_parsed


class MockQueryAgent:
    def generate(self, state: AgentState, need: SearchNeed) -> QueryPlan:
        if need.intent == "TASK_DISCOVERY":
            return QueryPlan(
                query_id=f"task-discovery-{uuid.uuid4().hex[:8]}", intent="TASK_DISCOVERY", top_k=8,
                queries=[
                    "프로젝트 구현 기능 요구사항 개발 업무 작업 목록",
                    "담당자가 수행해야 하는 업무 산출물 완료 기준",
                    "필수 구현 항목 작업 결과물 검수 기준",
                ],
            )
        terms = " ".join(need.missing_information)
        hint = need.task_hint or state["primary_document_title"] or "프로젝트 업무"
        return QueryPlan(
            query_id=f"{need.intent.lower()}-{uuid.uuid4().hex[:8]}",
            intent=need.intent,
            queries=[f"{hint} {terms} 요구사항 산출물 조건"],
        )


class RuleBasedExtractionAgent:
    def decide(self, state: AgentState) -> ExtractionDecision:
        stage = STAGES[state["stage_index"]]
        if stage["name"] == "FINAL_EXTRACTION":
            task_core = evidence_for_stage(state, "TASK_CORE")
            if not task_core:
                return ExtractionDecision(action="FINALIZE", rationale="직접 업무 근거 없음", tasks=[])
            all_chunks = state["evidence_chunks"]
            role_chunks = evidence_for_stage(state, "ASSIGNMENT_REQUIREMENT")
            condition_chunks = evidence_for_stage(state, "EXECUTION_CONDITION")
            missing = []
            if not role_chunks: missing += ["required_role", "required_skills"]
            if not condition_chunks: missing += ["effort_hours", "due_date", "priority"]
            task = ExtractedTask(
                task_id="task-001",
                title="Google OAuth 로그인 API 구현",
                description=task_core[0]["text"],
                required_role="백엔드 개발자" if role_chunks else None,
                required_skills=["Django", "OAuth 2.0"] if role_chunks else [],
                effort_hours=16.0 if condition_chunks else None,
                deliverables=["로그인 API"],
                missing_fields=missing,
                evidence_chunk_ids=[c["chunk_id"] for c in all_chunks],
            )
            return ExtractionDecision(action="FINALIZE", rationale="고정 절차 완료", tasks=[task])

        intent = stage["intent"]
        evidence = evidence_for_stage(state, intent)
        attempts = state["stage_attempts"].get(stage["name"], 0)
        if not evidence and attempts < state["max_attempts_per_stage"]:
            need = SearchNeed(
                stage=stage["name"],
                intent=intent,
                task_hint="" if intent == "TASK_DISCOVERY" else state["primary_document_title"],
                missing_information=stage["required"],
                reason="현재 단계 관련 근거 Chunk가 없음",
            )
            return ExtractionDecision(action="SEARCH", rationale=need.reason, search_need=need)
        return ExtractionDecision(action="ADVANCE", rationale="근거 확보 또는 검색 한도 도달")


## 3. Retriever Adapter

실제 연동 시 `InMemoryVectorRetriever.search()`와 동일한 계약으로 pgvector 또는 사내 검색 API를 구현한다. 문서 범위 필터는 검색 전 반드시 적용한다.


In [ ]:
class VectorRetriever(Protocol):
    def search(
        self, *, project_id: str, document_ids: list[str], queries: list[str],
        intent: Intent, top_k: int, exclude_chunk_ids: list[str]
    ) -> list[RetrievedChunk]: ...


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[0-9A-Za-z가-힣_.+-]+", text.lower()))


class InMemoryVectorRetriever:
    """노트북 검증용. 운영에서는 실제 VectorDB Adapter로 교체한다."""
    def __init__(self, chunks: list[RetrievedChunk]):
        self.chunks = chunks

    def search(self, *, project_id, document_ids, queries, intent, top_k, exclude_chunk_ids):
        query_tokens = tokenize(" ".join(queries))
        scored = []
        for chunk in self.chunks:
            if chunk.document_id not in document_ids or chunk.chunk_id in exclude_chunk_ids:
                continue
            overlap = len(query_tokens & tokenize(chunk.contextualized_text or chunk.text))
            intent_bonus = 3 if chunk.intent == intent else 0
            score = overlap + intent_bonus
            if score > 0:
                scored.append((score, chunk))
        scored.sort(key=lambda item: item[0], reverse=True)
        result = []
        for score, chunk in scored[:top_k]:
            copied = chunk.model_copy(update={"retrieval_score": min(0.99, 0.55 + score * 0.05)})
            result.append(copied)
        return result


In [ ]:
SAMPLE_CHUNKS = [
    RetrievedChunk(
        document_id="DOC-001", chunk_id="DOC-001:chunk:0003", sequence=3,
        text="본 기획서는 Google 계정 로그인과 인증 API 구현을 프로젝트 수행 범위로 정의한다.",
        contextualized_text="프로젝트 범위 > 핵심 기능\nGoogle 계정 로그인과 인증 API 구현",
        heading_path=["프로젝트 범위", "핵심 기능"], page_numbers=[1], intent="TASK_DISCOVERY"
    ),
    RetrievedChunk(
        document_id="DOC-001", chunk_id="DOC-001:chunk:0012", sequence=12,
        text="Google 계정으로 로그인할 수 있도록 OAuth 로그인 API를 구현한다. 산출물은 인증 API와 테스트 결과서다.",
        contextualized_text="인증 > 요구사항\nGoogle 계정 OAuth 로그인 API 구현 및 테스트 결과서",
        heading_path=["인증", "요구사항"], page_numbers=[3], intent="TASK_CORE"
    ),
    RetrievedChunk(
        document_id="DOC-001", chunk_id="DOC-001:chunk:0021", sequence=21,
        text="백엔드 개발자가 Django와 OAuth 2.0을 사용해 인증 기능을 담당한다.",
        contextualized_text="인증 > 구현 조건\n담당 역할 백엔드 개발자, 필수 기술 Django OAuth 2.0",
        heading_path=["인증", "구현 조건"], page_numbers=[5], intent="ASSIGNMENT_REQUIREMENT"
    ),
    RetrievedChunk(
        document_id="DOC-001", chunk_id="DOC-001:chunk:0030", sequence=30,
        text="인증 API 구현 예상 공수는 16시간이며 로그인 화면 연동 전에 완료한다.",
        contextualized_text="일정 > 인증\n예상 공수 16시간, 로그인 화면 연동 전 완료",
        heading_path=["일정", "인증"], page_numbers=[7], intent="EXECUTION_CONDITION"
    ),
]

retriever: VectorRetriever = InMemoryVectorRetriever(SAMPLE_CHUNKS)
query_agent: QueryAgent = OpenAIQueryAgent() if USE_OPENAI else MockQueryAgent()
extraction_agent: ExtractionAgent = OpenAIExtractionAgent() if USE_OPENAI else RuleBasedExtractionAgent()


## 4. StateGraph 노드와 라우팅


In [ ]:
def validate_selection_node(state: AgentState) -> dict[str, Any]:
    primary_id = state.get("primary_document_id", "").strip()
    if not primary_id:
        raise ValueError("사용자가 선택한 대표 기획서 primary_document_id가 필요합니다.")
    if primary_id not in state["document_ids"]:
        raise ValueError("primary_document_id는 허용된 document_ids에 포함되어야 합니다.")
    return {"trace": state["trace"] + [f"selection_validated:{primary_id}"]}


def extract_tasks_agent_node(state: AgentState) -> dict[str, Any]:
    decision = extraction_agent.decide(state)
    stage = STAGES[state["stage_index"]]
    trace = state["trace"] + [f"extract:{stage['name']}:{decision.action}"]

    if decision.action == "SEARCH":
        attempts = state["stage_attempts"].get(stage["name"], 0)
        if attempts >= state["max_attempts_per_stage"]:
            return {
                "stage_index": min(state["stage_index"] + 1, len(STAGES) - 1),
                "action": "ADVANCE",
                "warnings": state["warnings"] + [f"{stage['name']} 검색 한도 도달"],
                "trace": trace + ["forced_advance:max_attempts"],
            }
        if decision.search_need is None:
            raise ValueError("SEARCH 결정에는 search_need가 필요합니다.")
        return {"action": "SEARCH", "search_need": decision.search_need.model_dump(), "trace": trace}

    if decision.action == "ADVANCE":
        return {
            "action": "ADVANCE",
            "stage_index": min(state["stage_index"] + 1, len(STAGES) - 1),
            "search_need": None,
            "trace": trace,
        }

    return {
        "action": "FINALIZE",
        "extracted_tasks": [task.model_dump() for task in decision.tasks],
        "trace": trace,
    }


def query_generation_agent_node(state: AgentState) -> dict[str, Any]:
    need = SearchNeed.model_validate(state["search_need"])
    plan = query_agent.generate(state, need)
    used = {q.strip().lower() for q in state["query_history"]}
    unique_queries = [q for q in plan.queries if q.strip().lower() not in used]
    if not unique_queries:
        unique_queries = [f"{need.task_hint} {' '.join(need.missing_information)}"]
    plan = plan.model_copy(update={"queries": unique_queries[:3]})
    return {
        "query_plan": plan.model_dump(),
        "query_history": state["query_history"] + plan.queries,
        "trace": state["trace"] + [f"query:{plan.intent}:{plan.queries}"],
    }


def vector_search_node(state: AgentState) -> dict[str, Any]:
    plan = QueryPlan.model_validate(state["query_plan"])
    seen = [c["chunk_id"] for c in state["evidence_chunks"]]
    chunks = retriever.search(
        project_id=state["project_id"],
        document_ids=[state["primary_document_id"]] if plan.intent == "TASK_DISCOVERY" else state["document_ids"],
        queries=plan.queries, intent=plan.intent, top_k=plan.top_k, exclude_chunk_ids=seen,
    )
    attempts = dict(state["stage_attempts"])
    attempts[plan.intent] = attempts.get(plan.intent, 0) + 1
    return {
        "latest_chunks": [c.model_dump() for c in chunks],
        "stage_attempts": attempts,
        "trace": state["trace"] + [f"retrieve:{plan.intent}:{len(chunks)}"],
    }


def merge_evidence_node(state: AgentState) -> dict[str, Any]:
    merged = {c["chunk_id"]: c for c in state["evidence_chunks"]}
    for chunk in state.get("latest_chunks", []):
        merged[chunk["chunk_id"]] = chunk
    ranked = sorted(merged.values(), key=lambda c: c.get("retrieval_score", 0), reverse=True)
    ranked = ranked[: state["max_total_chunks"]]
    return {
        "evidence_chunks": ranked,
        "latest_chunks": [],
        "trace": state["trace"] + [f"merge:total={len(ranked)}"],
    }


def task_tools_node(state: AgentState) -> dict[str, Any]:
    valid_ids = {c["chunk_id"] for c in state["evidence_chunks"]}
    validated = []
    for raw in state["extracted_tasks"]:
        task = ExtractedTask.model_validate(raw)
        if not set(task.evidence_chunk_ids).issubset(valid_ids):
            raise ValueError(f"알 수 없는 근거 ID: {task.evidence_chunk_ids}")
        validated.append(task.model_dump())
    return {"extracted_tasks": validated, "trace": state["trace"] + [f"task_tools:{len(validated)}"]}


def calculate_workloads_node(state: AgentState) -> dict[str, Any]:
    workloads = [{"task_id": t["task_id"], "effort_hours": t.get("effort_hours")} for t in state["extracted_tasks"]]
    return {"workloads": workloads, "trace": state["trace"] + ["calculate_workloads"]}


def analyze_candidates_node(state: AgentState) -> dict[str, Any]:
    return {"candidate_results": [], "trace": state["trace"] + ["analyze_candidates:placeholder"]}


def finalize_node(state: AgentState) -> dict[str, Any]:
    result = {
        "tasks": state["extracted_tasks"],
        "workloads": state.get("workloads", []),
        "candidate_results": state.get("candidate_results", []),
        "warnings": state["warnings"],
        "evidence": state["evidence_chunks"],
        "trace": state["trace"],
    }
    return {"final_result": result, "trace": state["trace"] + ["finalize"]}


In [ ]:
def route_after_extractor(state: AgentState) -> str:
    return {"SEARCH": "query_generation_agent", "ADVANCE": "extract_tasks_agent", "FINALIZE": "task_tools"}[state["action"]]


def route_after_workload(state: AgentState) -> str:
    return "analyze_candidates" if state["extracted_tasks"] else "finalize"


builder = StateGraph(AgentState)
builder.add_node("validate_selection", validate_selection_node)
builder.add_node("extract_tasks_agent", extract_tasks_agent_node)
builder.add_node("query_generation_agent", query_generation_agent_node)
builder.add_node("vector_search", vector_search_node)
builder.add_node("merge_evidence", merge_evidence_node)
builder.add_node("task_tools", task_tools_node)
builder.add_node("calculate_workloads", calculate_workloads_node)
builder.add_node("analyze_candidates", analyze_candidates_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "validate_selection")
builder.add_edge("validate_selection", "extract_tasks_agent")
builder.add_conditional_edges(
    "extract_tasks_agent", route_after_extractor,
    {
        "query_generation_agent": "query_generation_agent",
        "extract_tasks_agent": "extract_tasks_agent",
        "task_tools": "task_tools",
    },
)
builder.add_edge("query_generation_agent", "vector_search")
builder.add_edge("vector_search", "merge_evidence")
builder.add_edge("merge_evidence", "extract_tasks_agent")
builder.add_edge("task_tools", "calculate_workloads")
builder.add_conditional_edges(
    "calculate_workloads", route_after_workload,
    {"analyze_candidates": "analyze_candidates", "finalize": "finalize"},
)
builder.add_edge("analyze_candidates", "finalize")
builder.add_edge("finalize", END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())


## 5. Mock 실행

실제 OpenAI 호출 전 그래프의 검색 루프, 단계 전환, 근거 연결, 종료를 검증한다.


In [ ]:
initial_state: AgentState = {
    "project_id": "PROJECT-001",
    "primary_document_id": "DOC-001",
    "primary_document_title": "프로젝트 인증 기능 기획서",
    "document_ids": ["DOC-001"],
    "analysis_goal": "프로젝트 수행 업무 추출",
    "stage_index": 0,
    "stage_attempts": {},
    "evidence_chunks": [],
    "query_history": [],
    "extracted_tasks": [],
    "warnings": [],
    "trace": [],
    "max_attempts_per_stage": MAX_ATTEMPTS_PER_STAGE,
    "max_total_chunks": MAX_TOTAL_CHUNKS,
}

result = graph.invoke(initial_state, {"recursion_limit": 30})
result["final_result"]


In [ ]:
assert result["final_result"]["tasks"], "직접 업무 근거가 있으므로 업무가 생성되어야 합니다."
assert all(t["evidence_chunk_ids"] for t in result["final_result"]["tasks"])
assert len(result["query_history"]) <= 3 * MAX_ATTEMPTS_PER_STAGE
assert result["stage_index"] == len(STAGES) - 1
print("검증 완료")
print("\n".join(result["trace"]))


## 6. 실제 연동 체크리스트

1. `OPENAI_API_KEY`를 환경 변수로 설정한다.
2. `USE_OPENAI=true`로 설정하고 커널을 재시작한다.
3. `InMemoryVectorRetriever`를 실제 pgvector/Retriever Adapter로 교체한다.
4. 사용자 UI에서 대표 기획서 하나를 선택받아 `primary_document_id`로 전달한다.
5. `TASK_DISCOVERY` 검색은 대표 기획서 하나로 제한하고, 후속 검색만 `document_ids` 범위를 사용한다.
6. 후보 분석과 업무 저장 노드를 기존 구현으로 교체한다.
7. 정상 빈 검색, VectorDB 장애, API 거절/타임아웃, 중복 질의, 검색 한도 테스트를 추가한다.
